In [1]:
import torch
import torch.nn as nn 
import os
import sys
project_root = '/public/home/zhangshikang/project/decouple_detr/DecoupleDETR/'
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)
from dataset import (build_dataset, build_loader,build_semi_weak_dataloader)
from models import build_model

/public/home/zhangshikang/.conda/envs/detr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/public/home/zhangshikang/.conda/envs/detr/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
check_point = '/public/home/zhangshikang/project/DATA/models/semi_train/for_test/epoch_49.pth'
model_dict = torch.load(check_point, map_location='cuda:0',weights_only=False)
model,criterion,postprocessor = build_model(model_dict['config'])
model.load_state_dict(model_dict['model_state_dict'])

/public/home/zhangshikang/.conda/envs/detr/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/public/home/zhangshikang/.conda/envs/detr/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<All keys matched successfully>

In [4]:
from util.config import ConfigDict
from eval.semi_eval import *
from tqdm import tqdm

### test on semi-dataset
- we use semi-dataset to test the model

In [5]:
cfg = ConfigDict.from_file('config/experiments/semi_train.yaml')
cfg.dataset.root = '/public/home/zhangshikang/project/DATA/semi_data/split/eval'
cfg.dataset.train_pct = 1.0
train_loader,_ = build_semi_weak_dataloader(cfg)

In [9]:
img,targets = next(iter(train_loader))
targets_  = targets.copy()

- process the targets and remove the boxes without cell labels

In [10]:
for t in targets_:
    labels = t['labels']
    t['boxes'] = t['boxes'][torch.where(labels!=-1)]
    t['labels'] = labels[torch.where(labels!=-1)]

In [13]:
CELL_LIST = ['T','Myeloid','Malignant','NK','Mast','Fibroblast','Epithelial','Endothelial','B','Plasma','SMC','Pericyte','Dendritic']

In [16]:
metrics = {
        #'map' : torchmetrics.detection.MeanAveragePrecision(box_format='cxcywh'),
        'f'   : SemiCellDetectionMetric(num_classes=13, 
                                    thresholds=0.2,
                                    max_pair_distance=12,
                                    class_names=CELL_LIST)
            }
### version 2
model = model.to('cuda')
for imgs,targets in tqdm(train_loader):
    imgs = imgs.to('cuda')
    for t in targets:
        labels = t['labels']
        t['boxes'] = t['boxes'][torch.where(labels!=-1)]
        t['labels'] = labels[torch.where(labels!=-1)]
    model.eval()
    outputs_ = {}
    with torch.no_grad():
        outputs = model(imgs,targets,stage='test',threshold=0.5)
        instance_logits = outputs['decouple_class_logits']
        instance_length = outputs['instance_length']
        instance_boxes = outputs['instance_boxes']
        for i, l in enumerate(instance_length):
            instance_boxes[i,l:,:2] = 2
            instance_logits[i,l:,:] = -100 
    outputs_['pred_logits'] = instance_logits.clone().cpu()
    outputs_['pred_boxes'] = instance_boxes.clone().cpu()
    orig_target_sizes = torch.stack([torch.tensor([256,256]) for t in targets], dim=0)
    predictions = postprocessor['bbox'](outputs_, orig_target_sizes)
    for p in predictions:
        # convert boxes
        p['boxes'] = box_ops.box_xyxy_to_cxcywh(p['boxes'])
    # prepare targets
    for t in targets:
        # get image size
        #img_h, img_w = data_loader.dataset.image_size(image_id=t["image_id"])
        img_h, img_w = 256,256
        # convert boxes
        t['boxes'] = box_ops.denormalize_box(t['boxes'], (img_h, img_w))
    # update metrics
    for k in metrics:
        metrics[k].update(predictions, targets)
metrics = {k: metrics[k].compute() for k in metrics}

100%|██████████| 270/270 [00:37<00:00,  7.21it/s]


In [17]:
metrics['f']

{'th02': {'detection': {'f1': 0.6126446077569535,
   'prec': 0.4619279919401877,
   'rec': 0.9093423799582463},
  'T': {'f1': np.float64(0.2230343531377258),
   'prec': np.float64(0.16219910395626091),
   'rec': np.float64(0.3568922305764411)},
  'Myeloid': {'f1': np.float64(0.20903851986654534),
   'prec': np.float64(0.1892990551527137),
   'rec': np.float64(0.2333739672219965)},
  'Malignant': {'f1': np.float64(0.1315319908233495),
   'prec': np.float64(0.10639175257731959),
   'rec': np.float64(0.17222963951935916)},
  'NK': {'f1': np.float64(0.03735881841876629),
   'prec': np.float64(0.07789855072463768),
   'rec': np.float64(0.02457142857142857)},
  'Mast': {'f1': np.float64(0.03176895306859206),
   'prec': np.float64(0.06451612903225806),
   'rec': np.float64(0.0210727969348659)},
  'Fibroblast': {'f1': np.float64(0.2627193466504801),
   'prec': np.float64(0.20446639467468328),
   'rec': np.float64(0.3673894590631993)},
  'Epithelial': {'f1': np.float64(0.27911988204604743),
   